# Credit Agreement Analyzer - Phase 4 Validation
Testing Q&A engine against the real Ribbon Communications credit agreement.

Prerequisites: Ollama running locally with `llama3:8b` loaded.

In [1]:
import contextlib
import time
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))

c:\Users\kbott\Projects\credit-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Rebuild Pipeline (Phase 1-3)
Re-run extraction through retrieval setup.

In [ ]:
# Phase 1: Extract, detect, parse, chunk
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)

defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
print(f"Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

Pages: 289 | Sections: 156 | Terms: 391 | Chunks: 710


In [3]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()}")
print(f"Available: {llm.is_available()}")

Provider: llama3.2:3b
Available: True


In [4]:
# Phase 3: Retrieval layer
embedder = Embedder()

start = time.time()
embeddings = embedder.embed([c.text for c in chunks])
print(f"Embedded {len(chunks)} chunks in {time.time() - start:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_p4")
store.create_collection("ribbon_p4")
store.add_chunks("ribbon_p4", chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)
print("Retrieval layer ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1287.57it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 710 chunks in 23.1s
Retrieval layer ready


## Step 1: Basic Q&A
Test single questions with no conversation history.

In [5]:
qa = QAEngine(retriever=retriever, llm=llm)
print("QAEngine initialized")

QAEngine initialized


In [6]:
def print_response(resp: QAResponse) -> None:
    """Display a QAResponse in a readable format."""
    print(f"Answer:\n{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  Section {s.section_id} - {s.section_title} (pp. {pages})")
        if s.relevant_text_snippet:
            print(f"    Snippet: {s.relevant_text_snippet[:120]}...")
    print(f"\nChunks retrieved: {len(resp.retrieved_chunks)}")
    print(f"LLM time: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used}")
    print("-" * 80)

In [7]:
# Q1: Total revolving commitment (exact dollar amount)
qa.clear_history()
print("Q: What is the total revolving commitment amount?\n")
resp = qa.ask("What is the total revolving commitment amount?", "ribbon_p4")
print_response(resp)

Q: What is the total revolving commitment amount?

Answer:
According to Section 2.4(a), the Total Revolving Commitments at any time are equal to the sum of (i) the Total Revolving Extensions of Credit and (ii) the L/C Exposure.

However, since we do not have enough information about the current Total Revolving Extensions of Credit or the L/C Exposure, I can only provide a general statement based on the initial amount of the Total Revolving Commitments.

The original amount of the Total Revolving Commitments is $35,000,000.

Confidence: LOW
Sources (5):
  Section 2.4 - Revolving Commitments (pp. 62)
    Snippet: 2.4 Revolving Commitments. Subject to the terms and conditions hereof, each Revolving Lender severally agrees to make re...
  Section 2.10 - Termination or Reduction of Commitments (pp. 64)
    Snippet: 2.10 Termination or Reduction of Commitments. The Borrower shall have the right, without penalty or premium, upon not le...
  Section 2.27 - Incremental Facility (pp. 83, 84, 85,

In [10]:
# Q2: Financinal covenants (list them)
qa.clear_history()
print("Q: What financial covenants are included in the agreement?\n")
resp = qa.ask("What financial covenants are included in the agreement?", "ribbon_p4")
print_response(resp)

Q: What financial covenants are included in the agreement?

Answer:
According to Section 7.1(b), the Borrower is required to comply with a specific covenant regarding its cash common equity contributions, which includes an "Equity Cure Expiration Date" (as defined in Section 7.1(a)).

Specifically, Section 7.1(a) states that if the Borrower fails to comply with Section 7.1(a) as of the last day of any period of four consecutive trailing fiscal quarters of Holdings, then:

"...any cash common equity contribution to the Borrower after the end of the applicable fiscal quarter and on or prior to the day that is ten Business Days after the day on which financial statements are required to be delivered for such fiscal quarter (the “Equity Cure Expiration Date”) will, at the irrevocable election of the Borrower, be included in the calculation of Consolidated Adjusted EBITDA solely for the purposes of determining compliance with Section 7.1(a) as of such date and as of any subsequent date that

In [8]:
# Q3: Incremental debt capacity (your core use case)
qa.clear_history()
print("Q: How much incremental debt can the borrower incur?\n")
resp = qa.ask("How much incremental debt can the borrower incur?", "ribbon_p4")
print_response(resp)

Q: How much incremental debt can the borrower incur?

Answer:
According to Section 2.27(a), the Borrower may request a Revolving Commitment Increase or an Incremental Term Facility with an aggregate principal amount not to exceed the Available Incremental Amount.

Confidence: HIGH
Sources (2):
  Section 2.27 - Incremental Facility (pp. 83, 84, 85, 86, 87)
    Snippet: 2.27 Incremental Facility.  (a) At any time after the Closing Date during the Commitment Period (in the case of a Revolv...
  Section 2.3 - Repayment of Term Loans (pp. 61, 62)
    Snippet: (ii) the rate of amortization (or the amount of any installment) with respect to any tranche of Loans may be increased (...

Chunks retrieved: 5
LLM time: 31.1s | Tokens: 135
--------------------------------------------------------------------------------


In [9]:
# Q4: Restricted payments (complex multi-basket answer)
qa.clear_history()
print("Q: What restricted payments are permitted under the credit agreement?\n")
resp = qa.ask("What restricted payments are permitted under the credit agreement?", "ribbon_p4")
print_response(resp)

Q: What restricted payments are permitted under the credit agreement?

Answer:
According to Section 7.6(k) of the Credit Agreement, the Group Members may make Restricted Payments (including with respect to Subordinated Indebtedness) in an aggregate amount not to exceed the Available Amount as in effect immediately before such Restricted Payment; provided that:

(i) no Event of Default has occurred and is continuing or would result therefrom
(ii) on a Pro Forma Basis, the Consolidated Net Leverage Ratio at the time of making such Restricted Payment shall be less than or equal to 1.90:1.00

Additionally, Section 7.6(k) also states that the Group Members may repurchase Capital Stock of Holdings pursuant to a board-approved share repurchase plan for aggregate consideration not to exceed $15,000,000.

Confidence: HIGH
Sources (3):
  Section 7.6 - Restricted Payments (pp. 123, 124, 125)
    Snippet: (a)  (i) any Subsidiary may make Restricted Payments to any Loan Party, and (ii) any Subsidia

In [11]:
# Q5: NOT FOUND test - question about something unlikely to be in the doc
qa.clear_history()
print("Q: What are the details of the borrower's pet insurance policy?\n")
resp = qa.ask("What are the details of the borrower's pet insurance policy?", "ribbon_p4")
print_response(resp)

Q: What are the details of the borrower's pet insurance policy?

Answer:
I could not find any information about the borrower's pet insurance policy in the provided sections. The relevant definitions section does not mention "pet insurance" or any similar term, and none of the sections related to Insurance (Section 4.24) provide details about the borrower's pet insurance policy.

Manual verification is recommended to determine the specifics of the borrower's pet insurance policy.

Confidence: LOW
Sources (5):
  Section 3.11 - Additional Issuing Lenders (pp. 95)
    Snippet: 3.11 Additional Issuing Lenders. The Borrower may, at any time and from time to time with the consent of the Administrat...
  Section 2.23 - Substitution of Lenders (pp. 80)
    Snippet: (a) a request from a Lender for payment of Indemnified Taxes or additional amounts under Section 2.20 or of increased co...
  Section 4.24 - Insurance (pp. 102)
    Snippet: 4.24 Insurance. All insurance maintained by the Loan Partie

## Step 2: Conversation Follow-ups
Test that question reformulation works for multi-turn Q&A.

In [12]:
qa.clear_history()

# Turn 1: Establish context
print("Turn 1: What is the term loan facility size?\n")
resp1 = qa.ask("What is the term loan facility size?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up using "it" (requires reformulation)
print("\nTurn 2: What is the interest rate on it?\n")
resp2 = qa.ask("What is the interest rate on it?", "ribbon_p4")
print_response(resp2)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What is the term loan facility size?

Answer:
I could not find this information in the sections I was able to retrieve from the agreement. You may want to check Section 2.27(a) manually for the definition of "Incremental Term Facility" and its relation to the available incremental amount.

However, based on the provided context, we can infer that the Incremental Facilities (Revolving Commitment Increase and/or Incremental Term Facility) are subject to an aggregate principal amount limit equal to the Available Incremental Amount.

Confidence: LOW
Sources (2):
  Section 2.27 - Incremental Facility (pp. 83, 84, 85, 86, 87)
    Snippet: 2.27 Incremental Facility.  (a) At any time after the Closing Date during the Commitment Period (in the case of a Revolv...
  Section 2.1 - Term Commitments (pp. 61)
    Snippet: 2.1 Term Commitments. Subject to the terms and conditions hereof, each Term Lender severally agrees to make a Term Loan ...

Chunks retrieved: 5
LLM time: 27.1s | Tokens: 9

In [13]:
qa.clear_history()

# Turn 1: Ask about covenants broadly
print("Turn 1: What financial covenants does this agreement have?\n")
resp1 = qa.ask("What financial covenants does this agreement have?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up referencing the previous answer
print("\nTurn 2: Are there any step-downs in those levels?\n")
resp2 = qa.ask("Are there any step-downs in those levels?", "ribbon_p4")
print_response(resp2)

# Turn 3: Another follow-up
print("\nTurn 3: What happens if the borrower breaches them?\n")
resp3 = qa.ask("What happens if the borrower breaches them?", "ribbon_p4")
print_response(resp3)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What financial covenants does this agreement have?

Answer:
According to Section 7.1, Financial Condition Covenants, there are two financial covenants:

1. Maximum Consolidated Net Leverage Ratio:
	* Commencing September 30, 2024, the Consolidated Net Leverage Ratio must not exceed 4.75:1.00 for each of the following four consecutive fiscal quarters:
		+ Four Consecutive Fiscal Quarter Ending
		+ September 30, 2024: 4.75:1.00
		+ December 31, 2024: 4.75:1.00
		+ March 31, 2025: 4.75:1.00
		+ June 30, 2025: 4.75:1.00
		+ September 30, 2025: 4.75:1.00
		+ December 31, 2025: 4.75:1.00
		+ March 31, 2026 and each fiscal quarter thereafter: 4.00:1.00

Confidence: HIGH
Sources (5):
  Section 9.15 - Certain ERISA Matters (pp. 143, 144)
    Snippet: (iv) such other representation, warranty and covenant as may be agreed in writing between the Administrative Agent, in i...
  Section 7.15 - Negative Pledge Clauses(f) (pp. 129)
    Snippet: 7.15 Negative Pledge Clauses(f). Enter into or su

## Step 3: Context Assembly Inspection
Look at the raw prompt being sent to the LLM to verify structure.

In [14]:
from credit_analyzer.generation.prompts import build_context_prompt

# Manually retrieve and build context to inspect
result = retriever.retrieve(
    query="What is the total revolving commitment amount?",
    document_id="ribbon_p4",
    top_k=5,
)

prompt = build_context_prompt(
    chunks=result.chunks,
    definitions=result.injected_definitions,
    history=[],
    question="What is the total revolving commitment amount?",
)

print(f"Prompt length: {len(prompt)} chars")
print(f"Approximate tokens: ~{len(prompt) // 4}")
print("\n" + "=" * 80)
print(prompt)
print("=" * 80)

Prompt length: 5806 chars
Approximate tokens: ~1451

=== CONTEXT FROM CREDIT AGREEMENT ===

--- Source: Defined Terms (Section 1.1, Pages 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56) ---
“Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment is a sublimit of the Total Revolving
Commitments.

--- Source: Defined Terms (Section 1.1, Pages 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56) ---
“Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth un

## Step 4: Timing & Token Budget
Verify we're within the 8K context window and check response times.

In [15]:
test_questions = [
    "What is the total revolving commitment amount?",
    "What are the financial covenant test levels?",
    "How much incremental debt can the borrower incur?",
    "Who is the administrative agent?",
    "What are the mandatory prepayment triggers?",
]

print(f"{'Question':<55} {'Time':>6} {'Tokens':>7} {'Conf':>6}")
print("-" * 80)

for q in test_questions:
    qa.clear_history()
    start = time.time()
    resp = qa.ask(q, "ribbon_p4")
    elapsed = time.time() - start
    print(
        f"{q:<55} {elapsed:>5.1f}s {resp.llm_response.tokens_used:>7} {resp.confidence:>6}"
    )

Question                                                  Time  Tokens   Conf
--------------------------------------------------------------------------------
What is the total revolving commitment amount?           32.1s     106    LOW
What are the financial covenant test levels?             30.3s     107   HIGH
How much incremental debt can the borrower incur?        32.1s     135   HIGH
Who is the administrative agent?                         19.3s      54   HIGH
What are the mandatory prepayment triggers?              38.1s     167   HIGH


## Cleanup

In [ ]:
store.delete_collection("ribbon_p4")
print("Cleaned up test collection")